In [ ]:
# Test if Running on Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Colab setup ────────────────────────────────────────────────────────────────────────────────────
if IN_COLAB:
    # Install the twin package directly from github
    import subprocess, os
    subprocess.run(["pip", "install", "--quiet", "--force-reinstall",
                   "git+https://github.com/alex-unofficial/twin-neighbor-embedding.git@experimental"], check=True)

    # Mount Google Drive for data access
    from google.colab import drive
    drive.mount('/content/drive')

    # Point DATA_DIR at Drive folder
    os.environ["DATA_DIR"] = "/content/drive/MyDrive/TWIN/data"
# ───────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neighbors import NearestNeighbors

import random
from PIL import Image
import seaborn as sns

from twin.config import DATA_DIR
from twin.embedding import SGtSNELayout
from twin.pipelines import graph_embedding_all

In [ ]:
n_obj = 20
n_image_per_obj = 72
n_samples = n_obj * n_image_per_obj

width = 64
height = 64
features = width * height

COIL_20_DIR = DATA_DIR / "point-cloud/coil-20"
COIL_20_IMAGE_DIR = COIL_20_DIR / "images"

In [ ]:
X = np.zeros((n_samples, features))
for i in range(n_obj):
    for j in range(n_image_per_obj):
        with Image.open(str(COIL_20_IMAGE_DIR / f"{i+1}/obj{i+1}__{j}.png")) as img:
            X[j + n_image_per_obj*i] = np.asarray(img.resize((width, height))).flatten()
            X[j + n_image_per_obj*i] /= np.max(X[j + n_image_per_obj*i])

In [ ]:
plt.imshow(X[random.randrange(0,n_samples)].reshape((height, width)))

In [ ]:
palette = sns.color_palette('hls', 20)
palette

In [ ]:
rng = np.random.default_rng(seed=0)
palette = rng.permutation(palette, axis=0)

In [ ]:
colors = np.repeat(palette, n_image_per_obj, axis=0)

In [ ]:
knn = NearestNeighbors(n_neighbors=12).fit(X)
adj = knn.kneighbors_graph(mode='distance').toarray()
G = nx.from_numpy_array(adj)

In [ ]:
experiment = graph_embedding_all(
    G, embedding=SGtSNELayout,
    twinmatrix_kw={'alpha': 0.85, 'k': 2},
    embedding_kw={'lambda_par': 10},
    seed=1
)

In [ ]:
y_v = experiment['VertexEmbedding']['V']
y_tw = experiment['TwinEmbedding']['V']

In [ ]:
plt.scatter(*y_v, s=1, c=colors)

In [ ]:
plt.scatter(*y_tw, s=1, c=colors)